In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

In [8]:
con = duckdb.connect(r'C:\Users\marzieh\Documents\GitHub\Tennis-project\data\tennis.duckdb', read_only=True)
con.execute("SHOW TABLES").df()

,name
0,_build_info
1,_import_audit
2,_schema_audit
3,_source_files
4,game_point_by_point
5,match_away_score
6,match_away_team
7,match_event
8,match_home_score
9,match_home_team


In [30]:
query = """
    WITH base_table AS (
        SELECT match_id, player_id, country, 1 as type
        FROM match_home_team
        Union
        SELECT match_id, player_id, country, 2 as type
        FROM match_away_team
    ),
    location AS (
        SELECT match_id, country AS match_location
        FROM match_venue
        WHERE match_location IS NOT NULL
    )
    SELECT DISTINCT b.match_id, b.player_id, b.country, b.type, e.winner_code, l.match_location
    FROM match_event AS e
    INNER JOIN base_table AS b
    ON e.match_id = b.match_id
    INNER JOIN location AS l
    ON e.match_id = l.match_id
    WHERE e.winner_code IS NOT NULL
    AND b.country IS NOT NULL
    AND b.country = l.match_location
"""
df = con.execute(query).df()
df

,match_id,player_id,country,type,winner_code,match_location
0,11998445,287803,France,1,2,France
1,11998675,237452,Australia,1,2,Australia
2,11998780,334293,USA,1,1,USA
3,11998782,36678,USA,1,1,USA
4,12026853,105375,Cyprus,1,2,Cyprus
...,...,...,...,...,...,...
4700,12174258,65576,Brazil,1,2,Brazil
4701,12200286,265808,Argentina,1,2,Argentina
4702,12211802,214236,Spain,1,1,Spain
4703,12211418,108563,Spain,2,1,Spain


In [33]:
duplicated_match_ids = df.loc[df["match_id"].duplicated(keep=False), "match_id"].unique()
clean_df = df[~df["match_id"].isin(duplicated_match_ids)].copy()
clean_df

,match_id,player_id,country,type,winner_code,match_location
0,11998445,287803,France,1,2,France
3,11998782,36678,USA,1,1,USA
4,12026853,105375,Cyprus,1,2,Cyprus
5,12023349,450685,China,1,1,China
6,12025687,162638,Hong Kong,1,1,Hong Kong
...,...,...,...,...,...,...
4697,12206198,101157,USA,1,2,USA
4698,12209122,420914,Japan,2,2,Japan
4702,12211802,214236,Spain,1,1,Spain
4703,12211418,108563,Spain,2,1,Spain


In [34]:
total_count = len(clean_df)
total_count

3311

In [37]:
win_count = (clean_df["type"] == clean_df["winner_code"]).sum()
win_count

np.int64(1816)

In [38]:
(win_count / total_count)*100

np.float64(54.847478103292055)